In [1]:
import pandas as pd

KAGGLE_PATH = '/kaggle/input/listings2025/listings25_CLEANED_FINAL.csv'

try:
    # Φόρτωση του αρχείου
    df_2025 = pd.read_csv(KAGGLE_PATH)
    print("Το αρχείο φορτώθηκε επιτυχώς!")
except FileNotFoundError:
    print(f"Σφάλμα: Το αρχείο δεν βρέθηκε. Ελέγξτε τη διαδρομή: {KAGGLE_PATH}")

Το αρχείο φορτώθηκε επιτυχώς!


In [2]:
# 1. Έλεγχος του τύπου δεδομένων (ΠΡΕΠΕΙ να είναι float64)
print(f"Τύπος δεδομένων στήλης 'price': {df_2025['price'].dtype}")

# 2. Έλεγχος των πρώτων τιμών (ΠΡΕΠΕΙ να είναι καθαροί αριθμοί χωρίς $)
print("Πρώτες 5 τιμές της στήλης 'price':")
print(df_2025['price'].head())

Τύπος δεδομένων στήλης 'price': object
Πρώτες 5 τιμές της στήλης 'price':
0        NaN
1     $30.00
2    $160.00
3     $50.00
4     $37.00
Name: price, dtype: object


In [3]:
# 1. Αφαιρούμε το σύμβολο $ και τυχόν κόμματα χιλιάδων
# Χρησιμοποιούμε τη στήλη 'price' του DataFrame
df_2025['price'] = df_2025['price'].astype(str).str.replace(r'[$,]', '', regex=True)

# 2. Μετατροπή σε αριθμητικό τύπο (float)
# Το errors='coerce' μετατρέπει τυχόν μη-αριθμητικές τιμές (όπως το 'nan' string) σε πραγματικά NaN.
df_2025['price'] = pd.to_numeric(df_2025['price'], errors='coerce')

print("Η στήλη 'price' καθαρίστηκε ξανά.")

Η στήλη 'price' καθαρίστηκε ξανά.


In [4]:
# Έλεγχος του Τύπου δεδομένων
print(f"Νέος τύπος δεδομένων στήλης 'price': {df_2025['price'].dtype}")

# Έλεγχος των τιμών (πρέπει να είναι αριθμοί)
print("Πρώτες 5 τιμές της καθαρισμένης στήλης 'price':")
print(df_2025['price'].head())

Νέος τύπος δεδομένων στήλης 'price': float64
Πρώτες 5 τιμές της καθαρισμένης στήλης 'price':
0      NaN
1     30.0
2    160.0
3     50.0
4     37.0
Name: price, dtype: float64


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [5]:
# 1. Βρίσκουμε τον συνολικό αριθμό NaN ανά στήλη
print("Αριθμός NaN ανά Στήλη (Απολυτός Αριθμός):")
nan_counts = df_2025.isnull().sum()
# Εμφανίζει μόνο τις στήλες με NaN, ταξινομημένες
print(nan_counts[nan_counts > 0].sort_values(ascending=False)) 

print("\n" + "="*50) # Διαχωριστικό

# 2. Βρίσκουμε το ποσοστό NaN ανά στήλη
print("Ποσοστό NaN ανά Στήλη (ως % των 9640 γραμμών):")
nan_percentage = df_2025.isnull().mean() * 100
# Εμφανίζει μόνο τις στήλες με NaN, ταξινομημένες κατά ποσοστό
print(nan_percentage[nan_percentage > 0].sort_values(ascending=False))

Αριθμός NaN ανά Στήλη (Απολυτός Αριθμός):
calendar_updated                14987
neighbourhood_group_cleansed    14987
host_neighbourhood               9296
host_about                       7546
neighborhood_overview            7447
neighbourhood                    7447
host_location                    4792
review_scores_accuracy           2572
first_review                     2572
reviews_per_month                2572
review_scores_value              2572
review_scores_communication      2572
review_scores_rating             2572
review_scores_cleanliness        2572
review_scores_checkin            2572
review_scores_location           2572
last_review                      2572
host_response_rate               1986
host_response_time               1986
beds                             1179
bathrooms                        1158
price                            1153
estimated_revenue_l365d          1153
host_is_superhost                 916
host_acceptance_rate              848
descript

In [6]:
columns_to_drop = [
    'calendar_updated', 
    'neighbourhood_group_cleansed', 
    'host_neighbourhood',
    'host_about',
    'neighborhood_overview',
    'license',
    'bathrooms' # Θα χρησιμοποιήσουμε το 'bathrooms_text'
]

df_2025 = df_2025.drop(columns=columns_to_drop, axis=1)
print(f"Διαγράφηκαν {len(columns_to_drop)} στήλες.")

Διαγράφηκαν 7 στήλες.


In [7]:
# 1. Διαγραφή γραμμών όπου η ΤΙΜΗ είναι NaN
df_2025= df_2025.dropna(subset=['price'])
print(f"Διαγράφηκαν {1153} γραμμές με NaN στην τιμή. Νέος αριθμός γραμμών: {len(df_2025)}")

# 2. Συμπλήρωση αριθμητικών στηλών με τη Διάμεσο (Median)
numerical_cols_to_fill = ['beds', 'bedrooms']
for col in numerical_cols_to_fill:
    df_2025[col] = df_2025[col].fillna(df_2025[col].median())

# 3. Συμπλήρωση Κριτικών & Ποσοστών με 0
review_cols = [col for col in df_2025.columns if 'review_scores_' in col or 'reviews_per_month' in col or 'rate' in col]
df_2025[review_cols] = df_2025[review_cols].fillna(0)

print("Ολοκληρώθηκε ο αριθμητικός καθαρισμός και η διαγραφή NaN τιμής.")

Διαγράφηκαν 1153 γραμμές με NaN στην τιμή. Νέος αριθμός γραμμών: 13834
Ολοκληρώθηκε ο αριθμητικός καθαρισμός και η διαγραφή NaN τιμής.


In [8]:
df_2025['neighbourhood'] = df_2025['neighbourhood'].fillna('Unknown')
df_2025['host_response_time'] = df_2025['host_response_time'].fillna('Unknown')
df_2025['host_is_superhost'] = df_2025['host_is_superhost'].fillna('f') # 'f' για False

print("Συμπληρώθηκαν οι κατηγορικές στήλες.")

Συμπληρώθηκαν οι κατηγορικές στήλες.


In [9]:
from datetime import datetime

TODAY = pd.to_datetime(datetime.now().date()) 
date_cols = ['host_since', 'first_review', 'last_review']

for col in date_cols:
    df_2025[col] = pd.to_datetime(df_2025[col], errors='coerce')

df_2025['host_age_days'] = (TODAY - df_2025['host_since']).dt.days
df_2025['review_span_days'] = (df_2025['last_review'] - df_2025['first_review']).dt.days
df_2025 = df_2025.drop(columns=date_cols)

print("4. Ολοκληρώθηκε το Feature Engineering των ημερομηνιών.")

4. Ολοκληρώθηκε το Feature Engineering των ημερομηνιών.


In [10]:
# Υποθέτουμε ότι το καθαρισμένο DataFrame σας ονομάζεται 'df_2025'
OUTPUT_FILE_PATH = '/kaggle/working/listings2025_cleaned_final.csv' # Ή listings2025_cleaned_final.csv

df_2025.to_csv(OUTPUT_FILE_PATH, index=False)

print(f"Το καθαρισμένο αρχείο αποθηκεύτηκε: {OUTPUT_FILE_PATH}")

# Πρέπει να πατήσετε Save Version/Commit στο Notebook για να εμφανιστεί το αρχείο στην Έξοδο (Output)

Το καθαρισμένο αρχείο αποθηκεύτηκε: /kaggle/working/listings2025_cleaned_final.csv


In [11]:
df_2025['last_scraped'] = pd.to_datetime(df_2025['last_scraped'], errors='coerce') 
df_2025['ETOS_LISTINGS'] = df_2025['last_scraped'].dt.year 
print("Το df είναι έτοιμο.")

Το df είναι έτοιμο.


In [12]:
df_2025['ETOS_LISTINGS'] = pd.to_numeric(df_2025['ETOS_LISTINGS'], errors='coerce').astype('Int64')
df_2025['ETOS_LISTINGS'] = pd.to_numeric(df_2025['ETOS_LISTINGS'], errors='coerce').astype('Int64')

print(f"Τύπος δεδομένων ETOS_LISTINGS (Combined): {df_2025['ETOS_LISTINGS'].dtype}")
print(f"Τύπος δεδομένων ETOS_LISTINGS (Macro): {df_2025['ETOS_LISTINGS'].dtype}")

Τύπος δεδομένων ETOS_LISTINGS (Combined): Int64
Τύπος δεδομένων ETOS_LISTINGS (Macro): Int64
